In [62]:
%pip install python-dotenv

Python(64339) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
34725.37s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [1]:
import csv
import requests
import time
import time
import pandas as pd
from dotenv import load_dotenv
import os

/Users/sooreoluwa/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# Load environment variables from the .env file
load_dotenv()

# Access the variables
COIN_MARKETCAP_API_KEY = os.getenv("COIN_MARKETCAP_API_KEY")
SEARCH_ENGINE_API_KEY = os.getenv("SEARCH_ENGINE_API_KEY")
SEARCH_ENGINE_ID = os.getenv("SEARCH_ENGINE_ID")

#### FETCH LEGITIMATE URLS DATA

In [4]:
coin_marketcap_base_url = "https://pro-api.coinmarketcap.com"

headers = {
    "Accepts": "application/json",
    "X-CMC_PRO_API_KEY": COIN_MARKETCAP_API_KEY,
}

def fetch_from_coin_marketcap():
    page = 1
    all_crypto = []
    max_pages = 20  
    limit = 1000

    while page <= max_pages:
        parameters = {
            "start": (page - 1) * limit + 1,
            "limit": limit,
            "convert": "USD"
        }

        try:
            response = requests.get(
                f"{coin_marketcap_base_url}/v1/cryptocurrency/listings/latest",
                headers=headers,
                params=parameters,
            )
            
            if response.status_code == 200:
                data = response.json()
                crypto_data = data.get("data", [])
                
                if not crypto_data:
                    break
                
                all_crypto.extend(crypto_data)
                page += 1
                
            elif response.status_code == 429:
                print("Rate limit hit. Waiting 60 seconds...")
                time.sleep(60)
                # Continue without incrementing page to retry

            else:
                print(f"Error {response.status_code}: {response.text}")
                break

        except requests.exceptions.RequestException as e:
            print(f"Request failed: {e}")
            break
    # Return the collected data        
    return all_crypto 

In [5]:
# function to fetch exchanges with pagination
def fetch_from_coingecko(url):
    page = 1
    all_exchanges = []
    
    while True:
        # Add the page parameter to the request
        params = {"per_page": 250, "page": page} 
        try:
            # Make the request
            response = requests.get(url, params=params)
            # Check if the request was successful
            if response.status_code == 200:
                data = response.json()
                # If no data is returned, stop the loop
                if not data:
                    break
                # Add the fetched data to the list
                all_exchanges.extend(data)
                print(f"Fetched page {page} with {len(data)} exchanges.")
                # Move to the next page
                page += 1
                # Add a delay to avoid hitting the rate limit
                time.sleep(1) 
    
            elif response.status_code == 429:
                # Handle rate limit error
                print("Rate limit exceeded. Waiting for 10 seconds before retrying...")
                time.sleep(60)  # Wait 60 seconds before retrying
            
            else:
                # Handle other errors
                print(f"Failed to fetch data. Status code: {response.status_code}. Reason: {response.reason}")
                break
        
        except requests.exceptions.RequestException as e:
            # Handle network errors
            print(f"Network error: {e}")
            break
    
    return all_exchanges

# Function to fetch exchanges from DeFi Llama
def fetch_from_other_source(url):
    # Make the request
    response = requests.get(url)
    if response.status_code == 200:
        # Return the JSON data
        return response.json() 
    else:
        print(f"Failed to fetch data. Status code: {response.status_code}. Reason: {response.reason}")
        
        return []
        

In [6]:
# function to convert to dictionary
def convert_to_dict(items):
    # Create a list to store the name-url mappings
    dict_list = []

    # Process each URL in the list
    for url in items:
        # Extract the text before the domain extension
        name = url.split('.')[0]
        # Create a dictionary for the name and URL
        dict_list.append({'name': name, 'url': url})

    return dict_list

# function to get the format a domain to URL
def format_domain(domain):
    if not domain.startswith(('https://', 'http://', 'www.')):
        return 'https://' + domain
    return domain

In [7]:
crypto_data_names = []

In [ ]:
# fetch centralized exchanges from coingecko
cex_url = "https://api.coingecko.com/api/v3/exchanges"
gecko_data = fetch_from_coingecko(cex_url)

for exchange in gecko_data:
    # Extract the name from the exchange data
    name = exchange['name']
    # Append the name to the list
    crypto_data_names.append(name)

# Print the total number of exchanges fetched
print(f"Total exchanges fetched: {len(gecko_data)}, {gecko_data[0]}")

In [ ]:
# /tags
# /tickers
# /coins
pap_exchanges = fetch_from_other_source("https://api.coinpaprika.com/v1/exchanges")

for exchange in pap_exchanges:
    # Extract the name from the exchange data
    name = exchange['name']
    # Append the name to the list
    crypto_data_names.append(name)

# Print the total number of exchanges fetched
print(f"Total exchanges fetched: {len(pap_exchanges)}, {pap_exchanges[0]}")

In [ ]:
# /tags
# /tickers
# /coins
pap_coins = fetch_from_other_source("https://api.coinpaprika.com/v1/coins")

for exchange in pap_coins:
    # Extract the name from the exchange data
    name = exchange['name']
    # Append the name to the list
    crypto_data_names.append(name)

# Print the total number of exchanges fetched
print(f"Total exchanges fetched: {len(pap_coins)}, {pap_coins[0]}")

In [ ]:
# fetch every exchange protocol from defi llama
# /chains
llama_url = "https://api.llama.fi/v2/protocols"
llama_data = fetch_from_other_source(llama_url)

for exchange in llama_data:
    # Extract the name from the exchange data
    name = exchange['name']
    # Append the name to the list
    crypto_data_names.append(name)

# Print the total number of exchanges fetched
print(f"Total exchanges fetched: {len(llama_data)}, {llama_data[0]}")

In [ ]:
coinmarketcap_data = fetch_from_coin_marketcap()

for exchange in coinmarketcap_data:
    # Extract the name from the exchange data
    name = exchange['name']
    # Append the name to the list
    crypto_data_names.append(name)

# Print the total number of exchanges fetched
print(f"Total exchanges fetched: {len(coinmarketcap_data)}, {coinmarketcap_data[0]}")

In [ ]:
len(crypto_data_names)

In [ ]:
def get_segment(my_list, start_index, segment_length=10000):
    # Calculate the end index
    end_index = start_index + segment_length
    
    # Ensure the end index does not exceed the list length
    if end_index > len(my_list):
        end_index = len(my_list)
    
    # Retrieve the segment
    return my_list[start_index:end_index]

In [ ]:
# Dictionary to store segments
segments_dict = {}

# Number of segments
num_segments = len(crypto_data_names) // 10000

# Retrieve and store each segment in the dictionary
for i in range(num_segments):
    start_index = i * 10000
    segment_key = f"Segment {i+1}"
    segments_dict[segment_key] = get_segment(crypto_data_names, start_index)

In [9]:
# Function to search Google for each exchange name
api_key = SEARCH_ENGINE_API_KEY
search_engine_id = SEARCH_ENGINE_ID

def search_google(query, wait_time=60):
    search_url = "https://www.googleapis.com/customsearch/v1"
    params = {
        'key': api_key,
        'cx': search_engine_id,
        'q': query,
        'num': 2
    }
    
    try:
        response = requests.get(search_url, params=params, timeout=10)
        if response.status_code == 200:
            results = response.json()
            if 'items' in results:
                links = [item['link'] for item in results['items']]
                return links
            else:
                print("No items found in the response.")
                return []
        elif response.status_code == 429:
            print(f"{response.reason}. Waiting for {wait_time} seconds before retrying...")
            time.sleep(wait_time)
            # After waiting, you can decide to call the function again if needed
            return search_google(query, wait_time)
        else:
            print(f"Error: {response.status_code} - {response.text}")
            return []
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
        return []

In [11]:
results = []

In [ ]:
for name in segments_dict['Segment 1']:
    urls = search_google(name)
    if urls:
        print(f"{name}: {urls}")  
        results.append({'name': name, 'url': urls})
    else:
        print(f"{name}: null")
        results.append({'name': name, 'url': 'null'})

In [ ]:
print(len(results))

In [ ]:
csv_file = 'legit_urls.csv'
with open(csv_file, mode='w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=['name', 'url'])
    writer.writeheader()
    writer.writerows(results)

print(f"Results saved to {csv_file}")

In [ ]:
# # fetch other legitimate websites asides crypto
# # to ensure balance in datasets.
# other_websites = "https://raw.githubusercontent.com/seigdev/resources/refs/heads/main/top-websites.csv"

# other_raw = fetch_data_csv(other_websites)

# others_df = pd.DataFrame(other_raw)

# others_df = others_df.drop(columns=0)

# others_df = others_df.rename(columns={1: 'url'})

In [ ]:
# df_raw = pd.read_json("legit-exchanges.json")

# # copy the name and urls columns to a new dataframe
# crypto_df = df_raw[['url']].copy()

# crypto_df["url"]  = crypto_df["url"].apply(format_domain)

# coinmarketcap_df = coinmarketcap_df[["url"]].copy()

# legit_df = pd.concat([crypto_df, coinmarketcap_df], ignore_index=True)

# # assign string labels to legitimate urls
# legit_df.loc[:, 'label'] = 'legit'

# # assign string labels to legitimate urls
# legit_df.loc[:, 'label_no'] = 0

# # select first 100
# legit_df = legit_df[:11000]

# # keep only the first occurrence
# legit_df = legit_df.drop_duplicates(subset="url", keep="first")

# legit_df

#### FETCH SCAM URLS DATA

In [ ]:
# url to fetch scam urls from eth-phishing-detect
scam_url = "https://raw.githubusercontent.com/MetaMask/eth-phishing-detect/master/src/config.json"
scam_ex = fetch_from_other_source(scam_url)

# fetch the list of blacklist urls
blacklist = scam_ex["blacklist"]

# convert the blacklist to a dictionary
blacklist = convert_to_dict(blacklist)

In [ ]:
with open("scam-exchanges.json", "w") as file:
    json.dump(blacklist, file)

blacklist_raw = pd.read_json("scam-exchanges.json")

# copy the name and urls columns to a new dataframe
scam_df = blacklist_raw[['url']].copy()

scam_df["url"] = scam_df["url"].apply(format_domain)

# assign string labels to legitimate urls
scam_df.loc[:, 'label'] = 'scam'

# assign string labels to legitimate urls
scam_df.loc[:, 'label_no'] = 1

# select first 100
# scam_df = scam_df.sample(n=100000, random_state=42).reset_index(drop=True)
scam_df = scam_df[:50000]

scam_df

In [ ]:
# merge both the legit and scam urls together
urls_df = pd.concat([legit_df, scam_df], ignore_index=True)

# shuffle the urls across the dataframe
urls_df = urls_df.sample(frac=1, random_state=42).reset_index(drop=True)

urls_df

In [ ]:
urls_df.to_json('crypto_data.json', orient='records', lines=False)